# Figure 3 — cross-QTL colocalization across molecular layers


This notebook reproduces **Figure 3** and its two supplementals. The figure asks
how often a genetic signal is **shared between molecular layers** — expression,
protein and splicing — using FastENLOC colocalization between each assay pair at
gene, credible-set (signal) and variant level, and whether that sharing depends on
allele frequency.

Figure 3's panels share a heavy upstream, so `00_setup.R` provides **memoized
derived-data builders** (`susie_finemapping()`, `coloc_gene_level()`,
`coloc_cs_level()`, `coloc_variant_level()`, `coloc_cs_components()`,
`coloc_model_input()`) that each build once and cache to `Fig3/data/`. Panels call
only the builders they need.


## How to run

* **Kernel:** R (IRkernel). **Working directory:** the repository root (the folder that contains `Fig3/`).
* Run the cells top to bottom. Each panel caches its ggplot object to `Fig3/objects/`, and the assembly cell reads those back.
* Re-running a single panel cell rebuilds just that panel; re-run the assembly cell to recompose.


### Inputs

> **Controlled-access.** Every input derives from *All of Us* controlled-access
> data and is not in this repository; run this on the Workbench.

Inputs are **manifest-driven** (`Fig3/inputs.tsv`): the AoU annotation table
auto-stages from its `gs://` URI, while the SuSiE and FastENLOC tables are marked
local-only and must already be staged. The derived-data builders cache to
`Fig3/data/` **by name**, so clear that directory (or pass `refresh = TRUE`) after
changing an input.

> **Results differ from the pre-cleanup figure.** Several fixes change output —
> most visibly Panel B, whose colocalization graph was previously built from one
> assay pair. See `Fig3/NOTES.md`.


## Setup

`00_setup.R` is the shared foundation every step builds on: it loads the libraries, sets the **Inter** font with a matched output **DPI**, defines the `theme_set`/`SZ_*` design system and the env-overridable paths, and brings in the manifest loaders and the folded-in analysis / plot functions plus the memoized derived-data builders — so the figure is self-contained.


In [ ]:
## Locate the figure directory. Terra / Linux is case-sensitive, so match
## either "Fig3" or "fig3" and point the pipeline's env-overridable
## paths at it before sourcing the shared setup.
fig_dir <- list.files(".", pattern = "^(Fig3|fig3)$", include.dirs = TRUE)
stopifnot("Run this notebook from the repository root (the folder holding Fig3/)." =
          length(fig_dir) == 1)
Sys.setenv(FIG3_MANIFEST = file.path(fig_dir, "inputs.tsv"),
           FIG3_OBJ = file.path(fig_dir, "objects"),
           FIG3_DATA = file.path(fig_dir, "data"))
source(file.path(fig_dir, "00_setup.R"))
cat("Setup loaded from", fig_dir, "\n")


### Panels

Each panel below **loads only the inputs it needs** (staging them on demand), does its wrangling, builds one ggplot object, caches it to `Fig3/objects/`, and renders it inline.


### Panel A — gene-level colocalization: percent of genes by the number of colocalized molecular layers, among genes fine-mapped in all three assays.

Among genes fine-mapped in **all three** assays, the percent of genes by the number of molecular layers that colocalize (0–3), from gene-level FastENLOC — how often a signal is shared across expression, protein and splicing. The '≥1 layer' figure in the annotation is computed from the data, not hard-coded.

*Inputs: enloc_gene, susie_eqtl, susie_sqtl, susie_pqtl*


In [ ]:
genes_all_three_assays <- susie_finemapping()$genes_all_three_assays

layer_counts <- coloc_gene_level() %>%
  filter(Gene %in% genes_all_three_assays) %>%
  group_by(Gene) %>%
  summarize(NumColocs = sum(Coloc), .groups = "drop") %>%
  dplyr::count(NumColocs) %>%
  mutate(prop = n / sum(n))

# Computed, never hardcoded: the original pasted "86.2%" into this annotation
# while computing the same value 130 lines earlier.
shared_pct <- layer_counts %>%
  filter(NumColocs > 0) %>%
  summarize(p = sum(prop)) %>%
  pull(p)

message(sprintf("Panel A: %.1f%% of genes share a signal across >=1 layer",
                100 * shared_pct))

pA <- layer_counts %>%
  mutate(NumColocs = factor(NumColocs, levels = 0:3)) %>%
  ggplot(aes(x = NumColocs, y = prop, fill = NumColocs)) +
    geom_col(width = 0.75) +
    scale_fill_manual(values = LAYER_FILL, guide = "none") +
    annotate("segment", x = 1.5, xend = 4.5, y = 0.455, yend = 0.455,
             color = "grey50", linewidth = 0.25) +
    annotate("text", x = 3, y = 0.465,
             label = sprintf("%.1f%% shared across ≥1 molecular layer",
                             100 * shared_pct),
             size = SZ_ANNOT, color = "grey20") +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                       breaks = seq(0, 0.5, by = 0.1), limits = c(0, 0.55)) +
    labs(x = "Number of Colocalized Molecular Layers", y = "Percent of Genes",
         title = "Gene-level Colocalization") +
    theme(axis.line.x = element_blank(),
          axis.ticks.x = element_blank(),
          plot.title = element_text(hjust = 0.5, size = SZ_TITLE, face = "bold"),
          plot.title.position = "plot")

saveRDS(pA, file.path(OBJ_DIR, "A.rds"))
message("Panel A built.")

pA


### Panel B — signal-level colocalization: percent of genes by the number of colocalized signals, with an inset of per-assay credible-set coloc rate.

Collapses colocalized credible sets into connected components (a signal shared across layers, via an undirected graph) and plots the percent of genes by number of colocalized signals, with an inset giving the per-assay-pair credible-set colocalization rate.

*Inputs: enloc_cs, susie_eqtl, susie_sqtl, susie_pqtl*


In [ ]:
cs_level        <- coloc_cs_components()
genes_all_three_assays <- susie_finemapping()$genes_all_three_assays

signal_counts <- cs_level %>%
  filter(!is.na(coloc_component)) %>%       # the original used na.omit()
  distinct(Gene, coloc_component) %>%
  dplyr::count(Gene) %>%
  dplyr::count(n, name = "nn")

main <- signal_counts %>%
  ggplot(aes(x = factor(n), y = nn / sum(nn), fill = factor(n))) +
    geom_col(width = 0.75) +
    scale_fill_manual(values = SIGNAL_FILL, guide = "none") +
    scale_y_continuous(labels = scales::percent_format(accuracy = 1),
                       limits = c(0, 0.55)) +
    labs(x = "Number of colocalized signals", y = "Percent of genes",
         title = "Signal-level Colocalization") +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank(),
          axis.title.y = element_blank(), axis.text.y = element_blank(),
          axis.ticks.y = element_blank(), axis.line.y = element_blank(),
          plot.title = element_text(hjust = 0.5, size = SZ_TITLE, face = "bold"),
          plot.title.position = "plot")

coloc_rate <- cs_level %>%
  filter(Gene %in% genes_all_three_assays) %>%
  group_by(type) %>%
  summarise(n_cs       = n_distinct(cs_id),
            n_coloc    = n_distinct(cs_id[has_coloc]),
            coloc_rate = n_coloc / n_cs,
            .groups = "drop")

inset <- coloc_rate %>%
  ggplot(aes(y = reorder(type, coloc_rate), x = coloc_rate, fill = type)) +
    geom_col(show.legend = FALSE) +
    paletteer::scale_fill_paletteer_d("ltc::minou") +
    scale_x_continuous(labels = scales::percent_format(accuracy = 1),
                       expand = c(0, 0)) +
    labs(x = "% CS coloc", y = NULL) +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank(),
          axis.line.y = element_line(color = "lightgrey"),
          axis.ticks.y = element_blank(),
          # No plot.tag = element_blank() override here: it would do nothing.
          # wrap_elements(full = ...) below folds `main` and this inset into
          # ONE opaque leaf before tagging happens, so recurse_tags() never
          # walks down into the inset to assign (or theme) a tag on it in the
          # first place -- confirmed empirically by rendering with `&
          # theme(plot.tag = element_text(color = "red"))` and observing no
          # tag, styled or not, on the inset either way.
          plot.margin = margin(0, 0, 0, 0))

pB_raw <- main + inset_element(inset, left = 0.48, bottom = 0.60,
                               right = 0.98, top = 0.98, align_to = "panel",
                               ignore_tag = TRUE)

# ignore_tag = TRUE keeps the inset itself out of the auto-tag sequence, but
# it has a sharp edge: patchwork gives "main + inset_element(...)" the class
# "inset_patch", and its auto-tagging walk (patchwork:::recurse_tags) skips
# ANY inset_patch object whose has_tag() is FALSE -- which, with ignore_tag =
# TRUE, is this whole composite, not just the inset. When pB_raw sits as a
# sibling among other panels (as it does once assemble.R does `P$A + P$B`),
# that means pB_raw's own main plot silently loses ITS tag too, and every
# later panel's letter shifts down by one -- confirmed empirically (see
# tests/test_fig3_assembly.R) before this wrap_elements() call was added.
# wrap_elements(full = ...) folds the whole composite into a single
# "wrapped_patch" leaf, which is not an inset_patch and so is exempt from
# that skip: it gets tagged normally as one panel, and the inset inside it
# still carries no tag of its own.
pB <- wrap_elements(full = pB_raw)

saveRDS(pB, file.path(OBJ_DIR, "B.rds"))
message("Panel B built.")

pB


### Panel C — effect of -log(MAF) on cross-assay effect-size correlation: the interaction coefficient and 95% CI for each assay pair.

Fits, per assay pair, the interaction between −log(MAF) and cross-assay effect-size correlation, and plots each coefficient with its 95% CI — does the strength of a shared signal track allele frequency? (The three per-pair model formulas are deliberately not identical; see `NOTES.md`.)

*Inputs: enloc_variant, susie_eqtl, susie_sqtl, susie_pqtl*


In [ ]:
# The original fit these models TWICE: once on the raw scale without an NA
# filter, then again with log1p() and NA filtering, discarding the first result.
# Only the second is kept.
#
# NOTE: the eQTL-pQTL model omits abs() on both posteriors, unlike the other two
# pairs. That asymmetry is preserved verbatim from the original; see NOTES.md,
# which flags it as possibly unintended (log1p() of a value below -1 is NaN).
FIT_FORMULA <- list(
  `eQTL-sQTL` = log1p(abs(posterior_mean.y)) ~ log1p(abs(posterior_mean.x)) * log_maf,
  `eQTL-pQTL` = log1p(posterior_mean.y)      ~ log1p(posterior_mean.x)      * log_maf,
  `sQTL-pQTL` = log1p(abs(posterior_mean.y)) ~ log1p(abs(posterior_mean.x)) * log_maf
)

fit_pair <- function(d, pair) {
  d <- filter(d, !is.na(posterior_mean.x), !is.na(posterior_mean.y))
  broom::tidy(lm(FIT_FORMULA[[pair]], data = d), conf.int = TRUE) %>%
    filter(!str_detect(term, "Int")) %>%
    mutate(type = pair)
}

interaction_res <- purrr::imap_dfr(coloc_model_input(), ~ fit_pair(.x, .y))

pC <- interaction_res %>%
  filter(str_detect(term, ":")) %>%
  ggplot(aes(x = estimate, y = type)) +
    geom_linerange(aes(xmin = conf.low, xmax = conf.high), color = "grey40") +
    geom_point(size = 4, color = "#364B63") +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey40") +
    labs(x = "Effect of -log(MAF) on cross-ome correlation", y = NULL) +
    theme(axis.line.y = element_blank(), axis.ticks.y = element_blank(),
          plot.margin = margin(l = 60, unit = "pt"))

saveRDS(pC, file.path(OBJ_DIR, "C.rds"))
message("Panel C built.")

pC


### Panel D — eQTL vs pQTL posterior effect size, faceted by MAF bin.

eQTL vs. pQTL fine-mapped effect sizes, faceted by MAF bin (Common / Low-Freq / Rare), with a per-bin slope and N — how expression and protein effects relate across the frequency spectrum.

*Inputs: enloc_variant, susie_eqtl, susie_sqtl, susie_pqtl*


In [ ]:
binned <- coloc_model_input()[["eQTL-pQTL"]] %>%
  filter(!is.na(max_af)) %>%
  mutate(maf_bin = case_when(max_af >= .05 ~ "Common",
                             max_af >= .01 ~ "Low-Freq",
                             TRUE          ~ "Rare"),
         posterior_mean.x = log1p(posterior_mean.x),
         posterior_mean.y = log1p(posterior_mean.y))

lims <- quantile(abs(c(binned$posterior_mean.x, binned$posterior_mean.y)),
                 0.97, na.rm = TRUE)

slopes <- binned %>%
  group_by(maf_bin) %>%
  summarise(n     = dplyr::n(),
            slope = coef(lm(posterior_mean.y ~ posterior_mean.x))[["posterior_mean.x"]],
            .groups = "drop") %>%
  mutate(label = paste0("β = ", round(slope, 2), "\nN = ", scales::comma(n)))

# Every row here is already Coloc == TRUE (coloc_model_input filters), so the
# original's redundant filter(Coloc == TRUE) inside geom_point is gone.
pD <- ggplot(binned, aes(x = posterior_mean.x, y = posterior_mean.y)) +
  geom_hline(yintercept = 0, linewidth = 0.25, color = "grey85") +
  geom_vline(xintercept = 0, linewidth = 0.25, color = "grey85") +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed",
              linewidth = 0.35, color = "grey65") +
  geom_point(alpha = 0.6, size = 4, color = "#6EC5C3") +
  geom_smooth(method = "lm", formula = y ~ x, se = TRUE, linewidth = 0.7,
              color = "red", fill = "red", alpha = 0.12) +
  geom_label(data = slopes, aes(x = -Inf, y = Inf, label = label),
             inherit.aes = FALSE, hjust = -0.05, vjust = 1.15,
             size = SZ_ANNOT, label.size = 0, fill = "white", alpha = 0.85) +
  facet_wrap(~maf_bin, nrow = 1) +
  coord_cartesian(xlim = c(-lims, lims), ylim = c(-lims, lims)) +
  labs(x = "Expression Posterior", y = "Protein Posterior") +
  theme(panel.spacing.x = unit(0.8, "lines"),
        plot.margin = margin(5.5, 8, 5.5, 5.5))

saveRDS(pD, file.path(OBJ_DIR, "D.rds"))
message("Panel D built.")

pD


### Panel E — reserved slot. The bottom row of Figure 3 leaves space for a panel that has not been decided yet (see fig3/NOTES.md). Rendering a blank keeps the layout stable and lets assemble.R stay unconditional.

A reserved slot in the bottom row; its content is undecided, so the code renders a blank to keep the layout stable (see `Fig3/NOTES.md`).


In [ ]:
pE <- blank_plot

saveRDS(pE, file.path(OBJ_DIR, "E.rds"))
message("Panel E built (reserved slot).")

pE


## Assemble the figure

Loads the cached panel objects from `Fig3/objects/` and composes them with patchwork into the final figure. Panel **tagging happens here and only here**. Writes the PDF plus a PNG preview, and renders the assembled figure inline.


In [ ]:
P <- setNames(lapply(LETTERS[1:5],
                     function(x) readRDS(file.path(OBJ_DIR, paste0(x, ".rds")))),
              LETTERS[1:5])

coloc_counts <- P$A + P$B + plot_layout(widths = c(1, 1))
top_row      <- (coloc_counts | P$C) + plot_layout(widths = c(2.5, 1))
bottom_row   <- (P$D | P$E) + plot_layout(widths = c(1.3, 1))

Fig3 <- (top_row / bottom_row) +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(size = SZ_TAG, face = "bold"))

ggsave(file.path(OUTDIR, "Figure3.pdf"), Fig3,
       width = 18, height = 12, dpi = DPI, limitsize = FALSE)
# Raster preview: full physical size at lower DPI (showtext DPI matched).
showtext_opts(dpi = 100)
ggsave(file.path(OUTDIR, "Figure3.png"), Fig3,
       width = 18, height = 12, dpi = 100, limitsize = FALSE)
showtext_opts(dpi = DPI)

message("Assembled: ", file.path(OUTDIR, "Figure3.pdf"), " and Figure3.png")

Fig3


## **Supplemental (colocalization QC).** GLCP / RCP / SCP posterior distributions by assay pair, over colocalization counts across posterior thresholds.


In [ ]:
gene_lvl    <- coloc_gene_level()
cs_lvl      <- coloc_cs_level()
variant_lvl <- coloc_variant_level()

density_panel <- function(d, col, show_legend) {
  ggplot(d, aes(x = .data[[col]], fill = type)) +
    # The original applied alpha = 0.7 only to the FIRST density panel
    # (GLCP); RCP and SCP used geom_density()'s opaque default. Applying a
    # uniform alpha to all three here is a deliberate, intentional deviation
    # from the original -- uniform alpha reads better across the row -- not
    # an oversight.
    geom_density(alpha = 0.7, show.legend = show_legend) +
    theme(axis.line.y = element_blank(), axis.ticks.y = element_blank(),
          axis.text.y = element_blank())
}

# wrap_elements() folds the three density panels into a single taggable
# element, so plot_annotation(tag_levels = "A") below sees this row as ONE
# leaf (tag "A") rather than walking into it and tagging each density panel
# separately (which would shift the discovery plot's tag from "B" to "D").
posterior_dists <- wrap_elements(full = density_panel(gene_lvl,    "GLCP", FALSE) +
                                         density_panel(cs_lvl,      "RCP",  FALSE) +
                                         density_panel(variant_lvl, "SCP",  TRUE))

THRESHOLDS <- c(0.01, 0.1, 0.5, 0.9)

discovery <- function(d, col, label) {
  purrr::map_dfr(THRESHOLDS, function(t) {
    d %>%
      group_by(type) %>%
      summarise(n_tested = dplyr::n(),
                value    = sum(.data[[col]] > t, na.rm = TRUE),
                .groups  = "drop") %>%
      mutate(threshold = t, posterior = label)
  })
}

discovery_by_type <- bind_rows(
  discovery(gene_lvl,    "GLCP", "GLCP"),
  discovery(cs_lvl,      "RCP",  "RCP"),
  discovery(variant_lvl, "SCP",  "SCP")
)

discovery_plot <- discovery_by_type %>%
  ggplot(aes(x = threshold, y = log10(value), group = type, color = type)) +
    geom_point() +
    geom_line() +
    facet_wrap(~posterior, nrow = 1) +
    labs(x = "Posterior threshold", y = "log10(Number Colocs)")

SuppColocQC <- posterior_dists / discovery_plot +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(size = SZ_TAG, face = "bold"))

ggsave(file.path(OUTDIR, "FigS_CrossQTLColocQC.pdf"), SuppColocQC,
       height = 7, width = 10, dpi = DPI)
message("Wrote ", file.path(OUTDIR, "FigS_CrossQTLColocQC.pdf"))

SuppColocQC


## **Supplemental (gene colocalization rate).** Proportion of genes colocalized, by assay pair.


In [ ]:
ProportionGeneColocs <- coloc_gene_level() %>%
  group_by(type) %>%
  summarize(PropColoc = mean(Coloc), .groups = "drop") %>%
  ggplot(aes(x = type, y = PropColoc)) +
    geom_col(position = "dodge") +
    labs(x = NULL, y = "Proportion Colocalized") +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank())

# Explicit plot = argument: the original relied on ggsave()'s last-plot default,
# which is correct in a notebook but fragile under Rscript.
ggsave(file.path(OUTDIR, "FigS_CrossQTLPropGeneColoc.pdf"), ProportionGeneColocs,
       height = 4, width = 6, dpi = DPI)
message("Wrote ", file.path(OUTDIR, "FigS_CrossQTLPropGeneColoc.pdf"))

ProportionGeneColocs
